# Лабораторная работа №10. Часть 1: Данные `trees`

**Задание:**
1. Выведите имена столбцов таблицы `trees`.
2. С помощью теста Шапиро-Уилка проверьте на нормальность каждый столбец таблицы. Сделайте выводы.
3. С помощью критерия Фишера проверьте гипотезы о независимости переменных. Сделайте выводы.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Создаем папку для картинок, если ее нет
os.makedirs("output", exist_ok=True)
ALPHA = 0.05

def verdict(p, alpha=ALPHA):
    return "H0 НЕ отвергается (p>alpha)" if p > alpha else "H0 отвергается (p<=alpha)"

### Загрузка данных
Используем локальный файл `trees.csv`, который лежит в папке с ноутбуком. Если его нет — скачиваем из Rdatasets.

In [ ]:
def load_data():
    local_file = "trees.csv"
    if os.path.exists(local_file):
        df = pd.read_csv(local_file)
    else:
        url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/trees.csv"
        df = pd.read_csv(url)
    # Убираем служебный столбец rownames, если он есть
    df = df.drop(columns=[c for c in df.columns if c.lower().startswith("row")], errors="ignore")
    return df

trees = load_data()

### 1. Имена столбцов

In [ ]:
print("[1] Имена столбцов таблицы trees:")
print(list(trees.columns))
trees.head()

### 2. Тест Шапиро-Уилка (нормальность)
**H0:** данные распределены нормально.  
**H1:** данные распределены не нормально.  
При `p > 0.05` нет оснований отвергать H0 (считаем нормальными).

In [ ]:
print("[2] Тест Шапиро-Уилка на нормальность")
print(f"{'Столбец':<10} | {'W':>8} | {'p-value':>10} | Вывод")
print("-" * 60)

sw_results = {}
for col in trees.columns:
    W, p = stats.shapiro(trees[col])
    sw_results[col] = (W, p)
    print(f"{col:<10} | {W:>8.4f} | {p:>10.4g} | {verdict(p)}")

**Вывод по п.2:**
* `Girth` (p = 0.089) и `Height` (p = 0.403) — H0 **не отвергается**, данные можно считать нормальными.
* `Volume` (p = 0.0036) — H0 **отвергается**, распределение значимо отличается от нормального.
* *Примечание для защиты:* Выборка маленькая (n=31). При малых n тест Шапиро-Уилка имеет низкую мощность (часто «пропускает» ненормальность). Отсутствие значимости ≠ доказанная нормальность. Кроме того, параметры при KS-тесте оцениваются по выборке (Lilliefors-проблема), поэтому Shapiro-Wilk предпочтительнее.

In [ ]:
# Визуализация: гистограммы для наглядности
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, trees.columns):
    ax.hist(trees[col], bins=10, edgecolor="k", alpha=0.7, color="skyblue")
    ax.set_title(f"{col}\n(Shapiro p = {sw_results[col][1]:.3g})")
    ax.set_xlabel("Значение")
    ax.set_ylabel("Частота")
plt.tight_layout()
plt.savefig("output/trees_hist.png", dpi=120)
plt.show()

### 3. Критерий Фишера (равенство дисперсий / независимость)
**H0:** дисперсии равны (σ₁² = σ₂²).  
**H1:** дисперсии различаются.  
*Примечание:* В задании написано «независимость», но F-тест Фишера проверяет именно **равенство дисперсий**. Для проверки истинной независимости (корреляции) мы дополнительно выведем матрицу корреляций.

In [ ]:
def f_test(a, b):
    """Двусторонний F-тест равенства дисперсий"""
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    F = a.var(ddof=1) / b.var(ddof=1)
    dfn, dfd = len(a) - 1, len(b) - 1
    cdf = stats.f.cdf(F, dfn, dfd)
    p = 2 * min(cdf, 1 - cdf)
    return F, p, dfn, dfd

print("[3] Критерий Фишера: попарное сравнение дисперсий")
cols = list(trees.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        a, b = trees[cols[i]], trees[cols[j]]
        F, p, dfn, dfd = f_test(a, b)
        print(f"  {cols[i]:<8} vs {cols[j]:<8} | F = {F:8.4f} | df = ({dfn}, {dfd}) | p = {p:.4g} | {verdict(p)}")

print("\nМатрица корреляций (для проверки истинной независимости):")
print(trees.corr().round(3))

**Вывод по п.3:**
* Все пары имеют `p < 0.05`, следовательно, H0 о равенстве дисперсий **отвергается**. Дисперсии значимо различаются (что логично, так как переменные измерены в разных единицах: дюймы, футы, кубические футы).
* **Про независимость:** Матрица корреляций показывает, что `Girth` и `Volume` сильно связаны (r = 0.967), а `Height` слабее связан с ними (r ~ 0.5). Это означает, что переменные **зависимы**.
* *Ограничение F-теста:* он очень чувствителен к нарушению нормальности (а `Volume` у нас не нормален). В таких случаях робастнее использовать тест **Levene** или **Bartlett**.